# Define Catalog and Schema

In [0]:
%sql
USE CATALOG github_lakehouse;
USE SCHEMA silver;

## Create the `push_events` Silver Table

This step creates the Delta table that will store the cleaned and structured **PushEvent** records in the Silver layer.

The table separates the fields into three groups:

- **Common event fields** — Core information shared across GitHub event types, such as the event ID, timestamp, actor, and repository.
- **PushEvent-specific fields** — Information specific to push activity, including the total number of commits and the number of distinct commits associated with the push.
- **Bronze lineage fields** — Metadata carried forward from the Bronze layer so each Silver record can be traced back to its original source file and ingestion.

The table is created using **Delta Lake** (`USING DELTA`) to support reliable storage and future data operations.

`CREATE TABLE IF NOT EXISTS` makes this step safe to rerun: if the table already exists, Databricks will not recreate or overwrite it.

> **Note:** This step only defines the Silver table structure. No Bronze data is inserted or transformed yet. Data extraction, cleaning, validation, and loading are handled in the following steps.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS silver.push_events (
    -- Common event fields
    event_id         STRING,
    event_type       STRING,
    event_timestamp  TIMESTAMP,
    event_date       DATE,
    actor_id         BIGINT,
    actor_login      STRING,
    repo_id          BIGINT,
    repo_name        STRING,

    -- PushEvent-specific fields
    push_commit_count           BIGINT,
    push_distinct_commit_count  BIGINT,

    -- Bronze lineage fields
    _source_file     STRING,
    _source_date     DATE,
    _ingested_at     TIMESTAMP,

    -- Quarantine Warning
    quarantine_warning STRING
)
USING DELTA;

## Prepare `PushEvent` Records from the Bronze Layer

This step reads the raw GitHub event data from the Bronze table and prepares the records required for the `PushEvent` Silver transformation.

The process is intentionally split into several stages so that parsing, lineage, and validation remain clear and traceable.

### 1. Read the Bronze source data

The Bronze table is loaded while keeping only the fields needed for this transformation:

- `raw_json` — The original GitHub event payload.
- `_source_file` — The source file from which the event was ingested.
- `_source_date` — The date associated with the source file.
- `_ingested_at` — The timestamp showing when the Bronze record was ingested.

These lineage fields are preserved so every Silver record can still be traced back to its original Bronze source.

### 2. Filter for `PushEvent` records

Before parsing the complete JSON payload, the code checks the event `type` directly from `raw_json` and keeps only records where:

`type = PushEvent`

Filtering first reduces unnecessary JSON parsing because the transformation only parses records relevant to this Silver table.

### 3. Define the required `PushEvent` source schema

A targeted schema is defined containing only the source fields needed for the Silver `push_events` table.

The schema includes:

- Event ID and event type
- Event creation timestamp
- Actor ID and login
- Repository ID and name
- Push commit count
- Distinct push commit count

The source values are initially read as **strings**, including numeric and timestamp fields.

This is intentional. Type validation and casting are handled later so invalid source values can be detected explicitly instead of being silently converted to null during parsing.

The schema also includes `_corrupt_record`, which provides a place to capture JSON records that cannot be parsed successfully.

### 4. Parse the raw JSON

`from_json()` converts the raw JSON string into a structured Spark column called `event`.

Parsing uses **PERMISSIVE** mode so problematic records do not immediately fail the entire transformation.

If Spark cannot correctly parse a record according to the supplied schema, the corrupt-record information can be retained for later validation and quarantine handling.

### 5. Extract the required fields

The required nested values are extracted from the parsed `event` structure and renamed to match the Silver table contract.

For example:

- `event.id` → `event_id`
- `event.created_at` → `event_timestamp`
- `event.actor.id` → `actor_id`
- `event.repo.name` → `repo_name`
- `event.payload.size` → `push_commit_count`
- `event.payload.distinct_size` → `push_distinct_commit_count`

At this stage, the DataFrame represents the logical structure expected for `PushEvent`, but validation and final datatype casting have **not yet been applied**.

### Validation evidence retained

Two additional fields are deliberately kept even though they are not part of the final Silver table:

- `raw_json` — Preserves the original source payload for debugging and quarantine.
- `_corrupt_record` — Identifies records that could not be parsed correctly.

These fields provide the evidence needed in later steps to validate records, identify contract violations, and separate invalid data from valid Silver records.

> **Note:** This step prepares and structures the source data only. Final datatype validation, business-rule checks, quarantine logic, and writing to the Silver table are performed in subsequent steps.

In [0]:
from pyspark.sql import functions as F

# 1. Read the Bronze data and preserve its lineage.
bronze_df = spark.table(
    "github_lakehouse.bronze.github_events_raw"
).select(
    "raw_json",
    "_source_file",
    "_source_date",
    "_ingested_at"
)


# 2. Filter PushEvent rows before parsing the full event.
push_bronze_df = bronze_df.filter(
    F.get_json_object(F.col("raw_json"), "$.type") == "PushEvent"
)


# 3. Define the source fields needed for PushEvent.
# Keep values as strings until the validation and casting step.
push_schema = """
STRUCT<
    id: STRING,
    type: STRING,
    created_at: STRING,
    actor: STRUCT<
        id: STRING,
        login: STRING
    >,
    repo: STRUCT<
        id: STRING,
        name: STRING
    >,
    payload: STRUCT<
        size: STRING,
        distinct_size: STRING
    >,
    _corrupt_record: STRING
>
"""


# 4. Parse the filtered JSON permissively.
parsed_df = push_bronze_df.withColumn(
    "event",
    F.from_json(
        F.col("raw_json"),
        push_schema,
        {
            "mode": "PERMISSIVE",
            "columnNameOfCorruptRecord": "_corrupt_record"
        }
    )
)


# 5. Extract the fields and prepare the PushEvent DataFrame.
push_events_df = parsed_df.select(
    F.col("event.id").alias("event_id"),
    F.col("event.type").alias("event_type"),
    F.col("event.created_at").alias("event_timestamp"),
    F.col("event.actor.id").alias("actor_id"),
    F.col("event.actor.login").alias("actor_login"),
    F.col("event.repo.id").alias("repo_id"),
    F.col("event.repo.name").alias("repo_name"),
    F.col("event.payload.size").alias("push_commit_count"),
    F.col("event.payload.distinct_size").alias(
        "push_distinct_commit_count"
    ),

    # Preserve the original Bronze lineage values.
    F.col("_source_file"),
    F.col("_source_date"),
    F.col("_ingested_at"),

    # Keep the raw evidence for later validation and quarantine.
    F.col("raw_json"),
    F.col("event._corrupt_record").alias("_corrupt_record")
)

## Validate and Convert `PushEvent` Data Types

This step converts the parsed `PushEvent` fields into the datatypes required by the Silver table while preserving the original source values for validation and quarantine.

The goal is to avoid unsafe casting. Instead of assuming that every source value is valid, the transformation keeps both the **converted value** and the **original raw value** so invalid records can be identified later.

### 1. Standardize timestamp handling

Spark is configured to use **UTC** for timestamp operations:

- `spark.sql.session.timeZone = UTC`
- `spark.sql.timestampType = TIMESTAMP_LTZ`

Using a consistent timezone ensures that GitHub event timestamps are interpreted and processed consistently throughout the pipeline.

### 2. Preserve and convert the event timestamp

Before converting `event_timestamp`, its original string value is copied into:

`event_timestamp_raw`

The timestamp is then converted using `try_to_timestamp()`.

Unlike a strict cast, `try_to_timestamp()` returns `NULL` when a value cannot be converted instead of failing the entire transformation.

This gives us two useful columns:

- `event_timestamp` — The converted timestamp.
- `event_timestamp_raw` — The original source value used for validation if conversion fails.

### 3. Derive the event date

`event_date` is derived from the successfully converted `event_timestamp`.

Using the validated timestamp rather than the raw string ensures that the Silver event date is based on a valid timestamp value.

### 4. Convert numeric fields safely

Fields that should be numeric are converted to `BIGINT` using `try_cast()`.

This includes:

- `actor_id`
- `repo_id`
- `push_commit_count`
- `push_distinct_commit_count`

For each converted field, the original source value is also retained in a corresponding `_raw` column.

For example:

- `actor_id` — Converted `BIGINT`
- `actor_id_raw` — Original source value
- `repo_id` — Converted `BIGINT`
- `repo_id_raw` — Original source value

The same pattern is applied to the PushEvent commit-count fields.

This allows later validation logic to distinguish between:

- a genuinely missing source value, and
- a value that was present but failed datatype conversion.

### 5. Pass valid string and lineage fields through unchanged

Fields that do not require datatype conversion are passed through directly, including:

- `event_id`
- `event_type`
- `actor_login`
- `repo_name`

The Bronze lineage fields are also preserved:

- `_source_file`
- `_source_date`
- `_ingested_at`

These fields maintain traceability from the Silver transformation back to the original Bronze record.

### 6. Preserve validation evidence

The transformation continues to retain:

- `raw_json`
- `_corrupt_record`

These fields are not intended to become part of the final clean Silver table. They remain temporarily available so later validation logic can inspect failed records and route invalid data to quarantine when necessary.

> **Note:** At the end of this step, the data has been safely typed, but records have not yet been classified as valid or invalid. The next validation stage can compare each converted field with its corresponding raw value, apply business rules, and determine which records are accepted into Silver or sent to quarantine.

In [0]:
from pyspark.sql import functions as F

# Use UTC for timestamps and the derived event date.
spark.conf.set("spark.sql.session.timeZone", "UTC")


# 1. Preserve the original timestamp, then convert it.
timestamp_df = (
    push_events_df
    .withColumn(
        "event_timestamp_raw",
        F.col("event_timestamp")
    )
    .withColumn(
        "event_timestamp",
        F.try_to_timestamp(F.col("event_timestamp"))
    )
)


# 2. Convert numeric fields and preserve their original values.
push_events_typed_df = timestamp_df.select(
    # String fields pass through directly.
    F.col("event_id"),
    F.col("event_type"),

    # Timestamp and its original value
    F.col("event_timestamp"),
    F.col("event_timestamp_raw"),

    # Derive the event date from the converted timestamp.
    F.expr("try_cast(event_timestamp AS DATE)").alias("event_date"),

    # Actor fields
    F.expr("try_cast(actor_id AS BIGINT)").alias("actor_id"),
    F.col("actor_id").alias("actor_id_raw"),
    F.col("actor_login"),

    # Repository fields
    F.expr("try_cast(repo_id AS BIGINT)").alias("repo_id"),
    F.col("repo_id").alias("repo_id_raw"),
    F.col("repo_name"),

    # Bronze lineage passes through directly.
    F.col("_source_file"),
    F.col("_source_date"),
    F.col("_ingested_at"),

    # Push counts and their original values
    F.expr("try_cast(push_commit_count AS BIGINT)").alias(
        "push_commit_count"
    ),
    F.col("push_commit_count").alias("push_commit_count_raw"),

    F.expr("try_cast(push_distinct_commit_count AS BIGINT)").alias(
        "push_distinct_commit_count"
    ),
    F.col("push_distinct_commit_count").alias(
        "push_distinct_commit_count_raw"
    ),

    # Evidence for later validation and quarantine
    F.col("raw_json"),
    F.col("_corrupt_record")
)

## Handle `PushEvent` NULLs and Quarantine Records

This step applies the Silver-layer NULL handling policy for `PushEvent` records.

The transformation:

- Converts blank string values into actual `NULL` values.
- Distinguishes between values that were genuinely missing in the source and values that became `NULL` because datatype conversion failed.
- Identifies **critical missing fields** such as `event_id`, `event_type`, and `repo_id`.
- Sends records with critical NULL issues to the `quarantine_push_event` table with a clear `quarantine_reason`.
- Keeps records with non-critical NULLs in the usable dataset.
- Adds a `quarantine_warning` column to record allowed missing values or conversion issues without rejecting the row.

At the end of this step, the data is separated into:

- `quarantine_push_df` — records that cannot safely continue through the Silver pipeline.
- `push_events_ready_df` — usable records that can continue to the next cleaning and validation steps.

> This step handles NULL-related data quality issues only. Other validation rules are applied separately.

In [0]:
from pyspark.sql import functions as F

QUARANTINE_TABLE = (
    "github_lakehouse.quarantine.quarantine_push_event"
)

spark.conf.set("spark.sql.session.timeZone", "UTC")

push_df = push_events_typed_df


# ---------------------------------------------------------
# 1. Normalize blank string fields to NULL
# ---------------------------------------------------------

for column_name in [
    "event_id",
    "event_type",
    "actor_login",
    "repo_name"
]:
    push_df = push_df.withColumn(
        column_name,
        F.when(
            F.col(column_name).isNull()
            | (F.trim(F.col(column_name)) == ""),
            F.lit(None)
        ).otherwise(
            F.col(column_name)
        )
    )


# ---------------------------------------------------------
# 2. Helper:
#    distinguish source NULL from failed conversion
# ---------------------------------------------------------

def conversion_null_reason(column_name):

    raw_column = F.col(column_name + "_raw")

    return F.when(
        F.col(column_name).isNull(),

        F.when(
            raw_column.isNull()
            | (F.trim(raw_column) == ""),

            F.lit("MISSING: " + column_name)

        ).otherwise(

            F.lit("INVALID_CAST: " + column_name)

        )
    )


# ---------------------------------------------------------
# 3. NULL conditions that require quarantine
# ---------------------------------------------------------

checked_df = push_df.withColumn(
    "quarantine_reason",

    F.concat_ws(
        "; ",

        F.when(
            F.col("event_id").isNull(),
            F.lit("MISSING: event_id")
        ),

        F.when(
            F.col("event_type").isNull(),
            F.lit("MISSING: event_type")
        ),

        conversion_null_reason("repo_id")
    )
)


# ---------------------------------------------------------
# 4. Split hard-null records from usable records
# ---------------------------------------------------------

quarantine_columns = (
    spark.table(QUARANTINE_TABLE).columns
)

quarantine_push_df = (
    checked_df
    .filter(
        F.col("quarantine_reason") != ""
    )
    .withColumn(
        "quarantined_at",
        F.current_timestamp()
    )
    .select(*quarantine_columns)
)


push_events_ready_df = (
    checked_df
    .filter(
        F.col("quarantine_reason") == ""
    )
    .drop("quarantine_reason")
)


# ---------------------------------------------------------
# 5. Record allowed NULLs / conversion-null warnings
# ---------------------------------------------------------

push_events_ready_df = (
    push_events_ready_df
    .withColumn(
        "quarantine_warning",

        F.concat_ws(
            "; ",

            conversion_null_reason("event_timestamp"),

            conversion_null_reason("actor_id"),

            F.when(
                F.col("actor_login").isNull(),
                F.lit("MISSING: actor_login")
            ),

            F.when(
                F.col("repo_name").isNull(),
                F.lit("MISSING: repo_name")
            ),

            conversion_null_reason(
                "push_commit_count"
            ),

            conversion_null_reason(
                "push_distinct_commit_count"
            ),

            F.when(
                F.col("_source_file").isNull(),
                F.lit("MISSING: _source_file")
            ),

            F.when(
                F.col("_source_date").isNull(),
                F.lit("MISSING: _source_date")
            ),

            F.when(
                F.col("_ingested_at").isNull(),
                F.lit("MISSING: _ingested_at")
            )
        )
    )
)


# Empty string -> actual NULL
push_events_ready_df = (
    push_events_ready_df
    .withColumn(
        "quarantine_warning",

        F.when(
            F.col("quarantine_warning") != "",
            F.col("quarantine_warning")
        )
    )
)

In [0]:
from pyspark.sql import functions as F


# ---------------------------------------------------------
# 1. Hard validation rules
#    Problems that make the row unsafe to treat as a valid PushEvent.
# ---------------------------------------------------------

# Rule 1: Spark could not fully parse the JSON for this row.
json_parse_error = F.when(
    F.col("_corrupt_record").isNotNull(),
    F.lit("JSON_PARSE_ERROR")
)

# Rule 2: defensive contract check - only PushEvent rows belong here.
# NULL event_type was already quarantined by the NULL cell, so skip it.
# invalid_event_type = F.when(
#     F.col("event_type").isNotNull()
#     & (F.col("event_type") != "PushEvent"),
#     F.lit("INVALID_EVENT_TYPE")
# )


# ---------------------------------------------------------
# 2. Non-fatal validation rules
#    The event stays usable; only a commit-volume metric is unreliable.
#    NULL counts were already warned about, so only check non-NULL values.
# ---------------------------------------------------------

# Rule 3: total commit count cannot be negative.
negative_commit_count = F.when(
    F.col("push_commit_count").isNotNull()
    & (F.col("push_commit_count") < 0),
    F.lit("INVALID_NEGATIVE_PUSH_COMMIT_COUNT")
)

# Rule 4: distinct commit count cannot be negative.
negative_distinct_commit_count = F.when(
    F.col("push_distinct_commit_count").isNotNull()
    & (F.col("push_distinct_commit_count") < 0),
    F.lit("INVALID_NEGATIVE_DISTINCT_COMMIT_COUNT")
)

# Rule 5: distinct commits are a subset of all commits in the push.
distinct_exceeds_total = F.when(
    F.col("push_commit_count").isNotNull()
    & F.col("push_distinct_commit_count").isNotNull()
    & (F.col("push_commit_count") >= 0)
    & (F.col("push_distinct_commit_count") >= 0)
    & (
        F.col("push_distinct_commit_count")
        > F.col("push_commit_count")
    ),
    F.lit("DISTINCT_COMMIT_COUNT_EXCEEDS_TOTAL")
)


# ---------------------------------------------------------
# 3. Apply the rules
#    concat_ws skips NULLs, so only failed rules appear in the text.
# ---------------------------------------------------------

push_validation_checked_df = (
    push_events_ready_df

    # All hard reasons for the row, e.g. "JSON_PARSE_ERROR; INVALID_EVENT_TYPE"
    .withColumn(
        "validation_quarantine_reason",
        F.concat_ws(
            "; ",
            json_parse_error,
            invalid_event_type
        )
    )

    # Existing NULL-cell warnings first, then new validation warnings.
    .withColumn(
        "quarantine_warning",
        F.concat_ws(
            "; ",
            F.col("quarantine_warning"),
            negative_commit_count,
            negative_distinct_commit_count,
            distinct_exceeds_total
        )
    )
)


# ---------------------------------------------------------
# 4. Empty string -> actual NULL
#    concat_ws returns "" when nothing failed.
# ---------------------------------------------------------

push_validation_checked_df = (
    push_validation_checked_df
    .withColumn(
        "validation_quarantine_reason",
        F.when(
            F.col("validation_quarantine_reason") != "",
            F.col("validation_quarantine_reason")
        )
    )
    .withColumn(
        "quarantine_warning",
        F.when(
            F.col("quarantine_warning") != "",
            F.col("quarantine_warning")
        )
    )
)


# ---------------------------------------------------------
# 5. Split hard failures from rows that continue
# ---------------------------------------------------------

# Hard failures: keep every column (raw_json, *_raw, _corrupt_record,
# warnings) as evidence for later quarantine finalization.
validation_quarantine_push_df = (
    push_validation_checked_df
    .filter(
        F.col("validation_quarantine_reason").isNotNull()
    )
)

# Rows that pass hard validation continue to deduplication.
# Temporary evidence columns are still kept on purpose.
push_events_validated_df = (
    push_validation_checked_df
    .filter(
        F.col("validation_quarantine_reason").isNull()
    )
    .drop("validation_quarantine_reason")
)